# JLoss Reconstruction v8 — versión 8


Reconstrucción del indicador de fragilidad sistémica bancaria $JLoss_{i,t}$ (Expected Loss + Unexpected Loss vía aproximación de punto de silla sobre un modelo de Merton).

**Certificación.** El motor `compute_jloss` reproduce los resultados oficiales de MATLAB (`CR` en `jloss.mat`) a tolerancia $<10^{-6}$ cuando el conjunto de bancos por trimestre coincide (verificado en Chile y Brasil sobre todos los trimestres; mediana exacta en Argentina, México y Egipto). Los residuos provienen exclusivamente de la reconstrucción de inputs, no del cálculo EL+UL.

**Estrategia de máxima fidelidad:**
1. **14 países** con `CR` válido (argentina, brazil, bulgaria, chile, china, colombia, egypt, indonesia, malaysia, mexico, pakistan, panama, philippines, poland): se toman **directamente** los resultados oficiales de MATLAB desde `jloss.mat`. *peru* figura en el mapeo pero su `CR` está vacío.
2. **4 países restantes** (russia, south_africa, turkey, venezuela), ausentes en `pd_indiv.mat`, se calculan desde cero con Merton/KMV + `compute_jloss` usando `mktcap_long.xlsx` y `balance_data_monthly.xls`.

**Parámetros de la corrida oficial** (verificados en `jloss.mat`): $\rho=0.4$ constante, $LGD=0.45$, multiplicidad de contraparte $=1$, percentil $=0.99$, malla de pérdida $[0.01,0.048]$ con 500 pasos, cuadratura de Gauss–Hermite de orden 7.

**Requisitos de ejecución:** colocar en el directorio de trabajo `jloss.mat`, `datesall_complete.xls`, `mktcap_long.xlsx`, `balance_data_monthly.xls` (y `pd_indiv.mat` sólo para la celda de validación). Dependencias: `scipy`, `pandas`, `numpy`, `matplotlib`, `seaborn`, `openpyxl`, `xlrd>=2.0.1`.

## 1. Imports y parámetros

In [ ]:
import scipy.io as sio
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve, brentq
from numpy.polynomial.hermite import hermgauss
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# --- parámetros del modelo (idénticos a la corrida oficial de MATLAB) ---
LGD=0.45; LOSS_INF=0.01; LOSS_SUP=0.048; NUM_STEPS=500
NUM_ORDER=7; PERCENTILE=0.99; R_FREE=0.04; MERTON_T=1.0
RHO_FLAT=0.4                       # rho oficial (rhos en jloss.mat es uniformemente 0.4)

def ghi_py(n=NUM_ORDER):
    x,w=hermgauss(n)
    return np.column_stack([np.sqrt(2)*x, w/np.sqrt(np.pi)])
GH_ZZ=ghi_py()

# slot en jloss.mat (variable CR) -> nombre de pais (los 15 oficiales)
SLOT_COUNTRY={0:'argentina',1:'brazil',2:'bulgaria',3:'chile',4:'china',
    5:'colombia',8:'egypt',11:'indonesia',14:'malaysia',15:'mexico',
    17:'pakistan',18:'panama',19:'peru',20:'philippines',21:'poland'}
MERTON_COUNTRIES={'russia','south_africa','turkey','venezuela'}

## 2. Motor JLoss (validado contra MATLAB)

Réplica de `countrypd.m`, `loss_distrib.m`, `loss_contrib.m`, `get_prob.m`, `LinealXY2.m`. La fórmula es $JLoss = (EL + UL)\times 100 / \sum EAD$, donde $UL$ son las contribuciones marginales al $VaR_{99}$ calculadas por punto de silla sobre exposiciones normalizadas. `find_var99` incorpora una guardia de monotonicidad de la CDF antes de interpolar.

In [ ]:
# ---------- Motor saddle-point (replica exacta de countrypd.m + dependencias) ----------
def cond_pd(pd_,rho,V):
    return float(norm.cdf((norm.ppf(np.clip(pd_,1e-8,1-1e-8))-rho*V)/np.sqrt(max(1-rho**2,1e-10))))
def K0(s,pc,a):
    t=0.0
    for p,ai in zip(pc,a):
        sa=s*ai
        if sa>500:t+=sa+np.log(max(p,1e-300))
        elif sa<-500:t+=np.log(max(1-p,1e-300))
        else:t+=np.log(1-p+p*np.exp(sa))
    return t
def K1(s,pc,a):
    t=0.0
    for p,ai in zip(pc,a):
        sa=s*ai
        if sa>500:t+=ai
        elif sa<-500:t+=p*ai
        else:
            e=np.exp(sa);t+=p*ai*e/(1-p+p*e)
    return t
def K2(s,pc,a):
    t=0.0
    for p,ai in zip(pc,a):
        sa=s*ai
        if abs(sa)>500:continue
        e=np.exp(sa);d=1-p+p*e;t+=p*e*ai**2/d-(p*e*ai/d)**2
    return t
def find_saddle(x,pc,a):
    try:
        lo=K1(-200,pc,a);hi=K1(200,pc,a)
        if x<=lo:return -200.0
        if x>=hi:return 200.0
        return brentq(lambda s:K1(s,pc,a)-x,-200,200,xtol=1e-9,maxiter=200)
    except:return np.nan
def get_prob_cdf(K0v,x,K2v,t):
    if K2v<=0:return 0.5
    lam=abs(t)*np.sqrt(K2v);exp_=K0v-t*x+0.5*t**2*K2v
    if exp_>700:temp=0.0
    elif exp_<-700:temp=1.0
    else:temp=np.exp(exp_)*norm.cdf(-lam)
    return float(np.clip(1.0-temp if t>0 else temp,0.0,1.0))
def loss_distrib_py(pds,rhos,a):
    N=len(pds);step=(LOSS_SUP-LOSS_INF)/NUM_STEPS
    xg=np.array([LOSS_INF+i*step for i in range(NUM_STEPS+1)]);pt=np.zeros(NUM_STEPS+1)
    for k in range(len(GH_ZZ)):
        Vk,wk=GH_ZZ[k,0],GH_ZZ[k,1]
        pc=np.array([cond_pd(pds[j],rhos[j],Vk) for j in range(N)])
        for idx,x in enumerate(xg):
            t=find_saddle(x,pc,a)
            if np.isfinite(t):pt[idx]+=wk*get_prob_cdf(K0(t,pc,a),x,K2(t,pc,a),t)
    return np.column_stack([xg,pt])
def find_var99(lp):
    xg=lp[:,0];cd=np.maximum.accumulate(lp[:,1])   # CDF monotona antes de interpolar
    if PERCENTILE<=cd[0]:return xg[0]
    if PERCENTILE>=cd[-1]:return xg[-1]
    idx=max(0,np.searchsorted(cd,PERCENTILE)-1);d=cd[idx+1]-cd[idx]
    if abs(d)<1e-10:return xg[idx]
    return xg[idx]+(xg[idx+1]-xg[idx])/d*(PERCENTILE-cd[idx])
def loss_contrib_py(exs,pds,rhos,a,lp):
    N=len(pds);nt=0.0;ct=np.zeros(N)
    for k in range(len(GH_ZZ)):
        Vk,wk=GH_ZZ[k,0],GH_ZZ[k,1]
        pc=np.array([cond_pd(pds[j],rhos[j],Vk) for j in range(N)])
        t=find_saddle(lp,pc,a)
        if not np.isfinite(t):continue
        K0v=K0(t,pc,a);K1v=K1(t,pc,a);K2v=K2(t,pc,a)
        if K2v<=0:continue
        ne=K0v-K1v*t
        if abs(ne)>700:continue
        nc=wk*np.exp(ne)/np.sqrt(K2v);nt+=nc
        for n in range(N):
            sa=t*a[n]
            if sa>500:tp=1.0
            elif sa<-500:tp=0.0
            else:
                en=np.exp(sa);tp=pc[n]*en/(1-pc[n]+pc[n]*en)
            ct[n]+=nc*tp
    if abs(nt)<1e-300:return np.zeros(N)
    return exs*ct/nt
def compute_jloss(pds,rhos,liabilities,lgd=LGD):
    pds=np.array(pds,float);rhos=np.array(rhos,float);liabilities=np.array(liabilities,float)
    v=(np.isfinite(pds)&np.isfinite(rhos)&np.isfinite(liabilities)&(liabilities>0)&(pds>0)&(pds<1)&(rhos>=0)&(rhos<1))
    pds,rhos,liabilities=pds[v],rhos[v],liabilities[v]
    if len(pds)<1:return np.nan
    exs=lgd*liabilities;exs1=exs*pds;a=exs/exs.sum();EL=float(exs1.sum())
    lp=loss_distrib_py(pds,rhos,a);loss_perc=find_var99(lp)
    UL=float(np.real(loss_contrib_py(exs,pds,rhos,a,loss_perc).sum()))
    return float((EL+UL)*100/liabilities.sum())

## 3. Parte A — 14 países desde `jloss.mat` (resultados oficiales MATLAB)

`CR[slot]` es una matriz $(52\times 3)$: columna 0 = $JLoss\%$. Se cruza con los trimestres de `datesall_complete.xls` (formato YYYYQ).

In [ ]:
# ---------- Parte A: 15 paises desde jloss.mat (resultados oficiales MATLAB) ----------
print("Parte A: leyendo CR de jloss.mat...")
CR = sio.loadmat('jloss.mat', simplify_cells=False, variable_names=['CR'])['CR']
datesall = pd.read_excel('datesall_complete.xls', header=None)[0].values.astype(int)
def yyyyq_to_period(d):
    d=int(d); return str(pd.Period(f"{d//10}Q{d%10}", freq='Q'))
rows_A=[]
for slot,country in SLOT_COUNTRY.items():
    cr=CR[0,slot]
    for ti in range(min(len(datesall),cr.shape[0])):
        jl=cr[ti,0]
        if np.isfinite(jl) and jl>0:
            rows_A.append({'countryname':country,'quarter':yyyyq_to_period(datesall[ti]),
                           'JLoss':float(jl),'source':'matlab'})
panel_A=pd.DataFrame(rows_A)
print(f"Parte A: {len(panel_A)} obs en {panel_A['countryname'].nunique()} paises")

## 4. Parte B — 4 países restantes vía Merton/KMV

Calibración fiel a `indivpds.m` / `KMVOptsearch.m` / `KMVfun.m`:

- Sistema de Merton en **variables normalizadas** $x=[V_A/E,\ \sigma_A]$ con $x_0=[1,1]$ (bien condicionado, reduce no-convergencias).
- **Distancia a default lineal de KMV** $DD=(V_A-DP)/(V_A\,\sigma_A)$, $EDF=\Phi(-DD)$ (no el $d_2$ de Black–Scholes).
- Anualización $\sigma_E=\sigma_{E,q}\sqrt{4}$; punto de default $DP=ST+0.5\,LT$; $\rho=0.4$ plano.

> Nota: `indivpds.m` contiene `DP=SD+05*LD` (evalúa como $5\cdot LD$, casi seguro un typo de $0.5$). Se mantiene $0.5\cdot LT$, el punto de default KMV canónico coherente con `KMVcompute_jf.m`.

In [ ]:
# ---------- Parte B: 4 paises restantes via Merton/KMV + compute_jloss ----------
# Calibracion fiel a indivpds.m / KMVOptsearch.m / KMVfun.m:
#   - variables normalizadas x=[Va/E, sigma_A], x0=[1,1]
#   - DD lineal de KMV: (Va-DP)/(Va*sigma_A); EDF=Phi(-DD)  (NO el d2 de Black-Scholes)
#   - sigma_E = sigma_E_q * sqrt(4)  (anualizacion, PriceTheta*sqrt(4))
#   - DP = ST + 0.5*LT ; rho = 0.4 plano (consistente con la corrida oficial)
def calc_merton_pd(E,sigma_E_q,D_star,r=R_FREE,T=MERTON_T):
    if not(np.isfinite(E) and np.isfinite(sigma_E_q) and np.isfinite(D_star)):return np.nan
    if E<=0 or sigma_E_q<=0 or D_star<=0:return np.nan
    sigma_E=sigma_E_q*np.sqrt(4.0);EtoD=E/D_star
    def kmvfun(x):
        v,sa=x
        if v<=1e-8 or sa<=1e-8:return[1e6,1e6]
        d1=(np.log(v*EtoD)+(r+0.5*sa**2)*T)/(sa*np.sqrt(T));d2=d1-sa*np.sqrt(T)
        return[v*norm.cdf(d1)-np.exp(-r*T)*norm.cdf(d2)/EtoD-1.0,norm.cdf(d1)*v*sa-sigma_E]
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            x,info,ier,_=fsolve(kmvfun,[1.0,1.0],full_output=True)
        v,sa=x
        if ier!=1 or v<=0 or sa<=0 or max(abs(np.array(kmvfun(x))))>1e-3:return np.nan
        Va=v*E;DD=(Va-D_star)/(Va*sa);edf=float(norm.cdf(-DD))
        return edf if 0.0<edf<1.0 else np.nan
    except Exception:return np.nan

print("Parte B: cargando Excel de mercado y balance...")
df_mkt=pd.read_excel('mktcap_long.xlsx')
df_mkt['date']=pd.to_datetime(df_mkt['date'],unit='D',origin='1899-12-30')
df_bal=pd.read_excel('balance_data_monthly.xls')
df_bal['date']=pd.to_datetime(df_bal['date'],unit='D',origin='1899-12-30')
df_mkt=df_mkt[df_mkt['countryname'].isin(MERTON_COUNTRIES)].copy()
df_bal=df_bal[df_bal['countryname'].isin(MERTON_COUNTRIES)].copy()

# vol realizada trimestral del equity (mktcap diario) y mktcap fin de trimestre
df_mkt=df_mkt.sort_values(['bankname','date'])
df_mkt['ret']=df_mkt.groupby('bankname')['mktcap'].pct_change()
df_mkt['quarter']=df_mkt['date'].dt.to_period('Q')
def rvol(g):
    r=g['ret'].dropna()
    return float(np.sqrt((r**2).sum())) if len(r)>=5 else np.nan
sigma_q=(df_mkt.groupby(['bankname','countryname','quarter'])
         .apply(rvol,include_groups=False).reset_index(name='sigma_E_q'))
mktcap_q=(df_mkt.groupby(['bankname','countryname','quarter'])['mktcap']
          .last().reset_index(name='mktcap_end'))

# balance -> DP y pasivos totales, ffill a malla trimestral
df_bal=df_bal.sort_values(['bankname','countryname','date'])
df_bal['quarter']=df_bal['date'].dt.to_period('Q')
allq=pd.period_range(df_bal['quarter'].min(),df_bal['quarter'].max(),freq='Q')
bks=df_bal[['bankname','countryname']].drop_duplicates()
full=pd.DataFrame([(b,c,q) for _,(b,c) in bks.iterrows() for q in allq],
                  columns=['bankname','countryname','quarter'])
balq=(df_bal.groupby(['bankname','countryname','quarter'])[['st_borrow','lt_borrow']]
      .last().reset_index())
balq=full.merge(balq,on=['bankname','countryname','quarter'],how='left').sort_values(['bankname','quarter'])
balq[['st_borrow','lt_borrow']]=(balq.groupby(['bankname','countryname'])
                                  [['st_borrow','lt_borrow']].transform(lambda x:x.ffill()))
balq['D_star']=balq['st_borrow']+0.5*balq['lt_borrow']
balq['total_liab']=balq['st_borrow']+balq['lt_borrow']
balq=balq.dropna(subset=['D_star','total_liab'])

dm=(sigma_q.merge(mktcap_q,on=['bankname','countryname','quarter'],how='inner')
    .merge(balq[['bankname','countryname','quarter','D_star','total_liab']],
           on=['bankname','countryname','quarter'],how='left'))
dm['PD']=dm.apply(lambda r:calc_merton_pd(r['mktcap_end'],r['sigma_E_q'],r['D_star']),axis=1)
print(f"Merton: {dm['PD'].notna().sum()} PDs validas de {len(dm)}")

rows_B=[]
for (country,quarter),g in dm.groupby(['countryname','quarter']):
    g=g[g['PD'].notna()&g['total_liab'].notna()&(g['total_liab']>0)]
    if len(g)<1:continue
    jl=compute_jloss(g['PD'].values,np.full(len(g),RHO_FLAT),g['total_liab'].values)
    if jl is not None and np.isfinite(jl):
        rows_B.append({'countryname':country,'quarter':str(quarter),
                       'JLoss':float(jl),'source':'merton'})
panel_B=pd.DataFrame(rows_B)
print(f"Parte B: {len(panel_B)} obs en {panel_B['countryname'].nunique()} paises")

## 5. Combinar y exportar

In [ ]:
# ---------- Combinar y exportar ----------
panel=pd.concat([panel_A,panel_B],ignore_index=True)
panel['date']=pd.PeriodIndex(panel['quarter'],freq='Q').to_timestamp()
panel=panel.sort_values(['countryname','date']).reset_index(drop=True)
panel.to_csv('Panel_JLoss_v8.csv',index=False)
print(f"\nPanel final v8: {len(panel)} obs en {panel['countryname'].nunique()} paises")
print(panel.groupby('source')['JLoss'].count().to_string())
print(panel.groupby('countryname')['quarter'].count().sort_values().to_string())

## 6. Visualización

In [ ]:
# ---------- Visualizacion ----------
paises=sorted(panel['countryname'].unique())
n_cols=3;n_rows=(len(paises)+n_cols-1)//n_cols
fig,axes=plt.subplots(n_rows,n_cols,figsize=(n_cols*5,n_rows*4))
plt.suptitle('JLoss over Time for Each Country',fontsize=16,y=1.02)
axes=axes.flatten()
for ax,country in zip(axes,paises):
    g=panel[panel['countryname']==country]
    sns.barplot(data=g,x=g['quarter'].astype(str),y='JLoss',color='skyblue',ax=ax)
    ax.set_title(country.replace('_',' ').capitalize(),fontweight='bold')
    ax.set_xlabel('Year.Quarter');ax.set_ylabel('JLoss')
    ax.tick_params(axis='x',rotation=90,labelsize=6)
for ax in axes[len(paises):]:ax.set_visible(False)
plt.tight_layout(rect=[0,0.03,1,0.98])
plt.savefig('JLoss_by_country_v8.png',dpi=150,bbox_inches='tight')
print("Grafico guardado: JLoss_by_country_v8.png")

## 7. Validación del motor contra MATLAB (requiere `pd_indiv.mat`)

Reconstruye los inputs de un país desde `pd_indiv.mat` y compara `compute_jloss` (con $\rho=0.4$) contra el `CR` oficial. Demuestra fidelidad a nivel de motor para el informe de tesis.

In [ ]:
mat=sio.loadmat('pd_indiv.mat',simplify_cells=False,
                variable_names=['EDF','alldata_array_quarter'])
EDF_var=mat['EDF'];aq=mat['alldata_array_quarter']

def build_inputs(slot,td):
    p=EDF_var[0,slot][0,1][0,0];aq_p=aq[0,slot][0,1][0,0];nb=p.shape[1]
    pds,liabs=[],[]
    for k in range(nb):
        be=p[1,k];ba=aq_p[1,k]
        d2e=dict(zip(be[:,0],be[:,1]));d2s=dict(zip(ba[:,0],ba[:,5]));d2l=dict(zip(ba[:,0],ba[:,4]))
        edf=d2e.get(td,np.nan);st=d2s.get(td,np.nan);lt=d2l.get(td,np.nan)
        if np.isnan(edf) or edf<=0 or np.isnan(st) or np.isnan(lt) or st+lt<=0:continue
        pds.append(edf);liabs.append(st+lt)
    return pds,liabs

for slot,name in [(3,'chile'),(1,'brazil'),(8,'egypt')]:
    diffs=[]
    for ti in range(len(datesall)):
        cr=CR[0,slot][ti,0]
        if not np.isfinite(cr) or cr<=0:continue
        pds,liabs=build_inputs(slot,datesall[ti])
        if len(pds)<1:continue
        jl=compute_jloss(pds,np.full(len(pds),RHO_FLAT),liabs)
        if np.isfinite(jl):diffs.append(abs(jl-cr))
    d=np.array(diffs)
    print(f'{name:<8} n={len(d):3d}  max|diff|={d.max():.6f}  mediana={np.median(d):.6f}')